### 1.1.1. Flat vs Indexed Sets
#### General Syntax
```ampl
set SET_NAME [indexing] [attributes];
```

- `set` — AMPL keyword for declaring a set.
- `SET_NAME` — name of the set.
- `{indexing}` — optional indexing expression; creates an indexed family of sets, one set per index.
- `[attributes]` — optional restrictions, ordering, definition, or default value.


In accordance with the general syntax, AMPL set declarations can be divided into two common forms: simple sets and indexed families of sets.
| Type | AMPL syntax | Meaning |
|---|---|---|
| **Simple set** | `set PROD;` | Declares one set named `PROD`. |
| **Indexed family of sets** | `set ORIG {PROD};` | Declares a collection of sets: one `ORIG[p]` for each member `p` in `PROD`. |

#### Flat Sets

A **flat set** is a single AMPL set, not an indexed family of sets. You refer to it directly by name, for example `PROD`, `NODES`, or `ARCS`, not as `ARCS[i]`.

Flat does **not** mean only one-dimensional. A flat set can contain either single values or tuples.

| Flat set form | Example | Meaning |
|---|---|---|
| One-dimensional set | `set PROD;` | One collection of product names. |
| Tuple set | `set ARCS dimen 2;` | One collection of pairs `(i,j)`. |
| Restricted tuple set | `set ARCS within NODES cross NODES;` | One collection of valid node-to-node arcs. |

Flat sets are useful when all members belong to one common collection and there is no parent index. They are especially natural for **symmetric relationships**, such as allowed routes, arcs, or origin-destination pairs.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/Mihailby/AMPLKnowledge/blob/main/examples/1_1_1__1.ipynb)

In [ ]:
set PROD;                      # basic (scalar) set

set ARCS dimen 2;              # 2D set: elements are pairs (i,j)

set TIME ordered;              # ordered set (keeps sequence)
set TIME2 within TIME;         # ordered subset of TIME

# Ordered by expression
set SORTED_ITEMS ordered;      # index set for parameter
set ITEMS ordered by SORTED_ITEMS;    # ITEMS ordered by SORTED_ITEMS value  

set DAYS circular;             # ordered set with wrap-around

# Domain restriction
set A;
set B within A;                # B ⊆ A

# Initialization
set COLORS = {'red','blue','green'};  # explicitly assigned elements
set OPTIONS default {'A','B'}; # default value if no data given

#### Indexed Sets
An **indexed set** (collection of sets) is a set-valued construct that is **parameterized by an index set**. Formally, it can be viewed as a mapping from each element of an indexing set to a corresponding component set.
While a simple (scalar) set represents a single collection of elements, an indexed set defines a family of sets, where each index value selects its own component set. That is, for an indexing set ARCS, the declaration:

```python
set ARCS {NODES}
``` 

The declaration `set ARCS {i in NODES} within NODES;` defines an indexed family of sets: for each node `i`, `ARCS[i]` is a set of nodes reachable from `i`. Nothing requires `ARCS[i]` and `ARCS[j]` to have the same members or the same cardinality, so the structure is fully ragged.

Another common pattern arises in multi-product models:

```python
set DEST {PROD};
```
where `DEST[p]` is the set of destinations to which product `p` can be shipped.

`Indexed sets` are the natural way to represent **ragged (irregular) structures**, where the size or composition of inner sets varies across index values. They are especially useful for modelling **hierarchical (parent–child) relationships**, in contrast to **symmetric relationships**, which are typically represented using flat multidimensional (tuple) sets such as:

```python
set ARCS ⊆ NODES cross NODES;
```

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/Mihailby/AMPLKnowledge/blob/main/examples/1_1_1__2.ipynb)

In [ ]:
# Parameters linking products to nodes
set ORIG {PROD} within NODES;   # origin node per product: 
                                # If ORIG[p] is not in NODES, AMPL rejects the data because it must be within NODES.
set DEST {PROD} within NODES;   # destination node per product: 
                                # If DEST[p] is not in NODES, AMPL rejects the data because it must be within NODES.

# Global arc set
set ARCS dimen 2 within NODES cross NODES;  # valid arcs (i,j).
                                            # Each arc must have exactly 2 components.
                                            # Both components must be members of NODES.

# 1. Minimal indexed set
set AREA {p in PROD};   # set per product
                        # There is no "within ..." restriction here.
                        # So AREA[p] can contain almost any symbolic values.         

# 2. Indexed subset of arcs
set OUT_ARCS {i in NODES} within ARCS;  # subset of valid arcs assigned to node i;
                                        # AMPL checks membership in ARCS, but does not automatically check that the arc starts at i

# 3. Symbolically defined set (per product)
set LINKS {p in PROD} = ORIG[p] cross DEST[p];  # LINKS[p] is calculated automatically by AMPL.
                                                # It is the Cartesian product of ORIG[p] and DEST[p].

# 4. Default empty sets
set AREA2 {p in PROD} default {};   # AREA2[p] is empty unless you give data for it.

# 5. Indexed set with restricted members
set S {i in NODES} within PROD; # S[i] is a set of products associated with node i. The index must be from NODES. 

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Do not confuse the indexed collection <code>set ARCS {NODES}</code> with the flat 2-D set
<code>set ALL_ARCS within NODES cross NODES</code>. The first declares one set per node;
the second declares a single flat set of pairs. Writing <code>ARCS cross NODES</code> is a
syntax error because <code>ARCS</code> is an indexed collection, not a set expression.
To flatten an indexed family to a set of pairs, use
<code>setof {i in NODES, j in ARCS[i]} (i,j)</code>.</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
An indexed set family declared with <code>=</code> cannot receive data from a
<code>set</code> statement in a <code>.dat</code> file or a <code>let</code> command.
Attempting to assign data to <code>LINKS</code> (declared with
<code>= ORIG[p] cross DEST[p]</code>) produces an AMPL error. If the set must be
overridable, use <code>default</code> instead of <code>=</code>.</div>

#### When to use indexed families vs. flat tuple sets

| Criterion | Indexed family `set F {I}` | Flat tuple set `set T within I cross U` |
|---|---|---|
| Hierarchical (I is "parent" of U) | Natural | Awkward |
| Symmetric / peer relationship | Awkward | Natural |
| Need `F[i]` repeatedly in the model | One subscript | Slice syntax each time |
| Need all `(i,j)` pairs at once | Use `setof` | Direct reference |

The AMPL book (Fourer, Gay, Kernighan, Chapter 6) observes: *"indexed collections are
most suitable for entities such as products and areas that have a hierarchical
relationship. Sets of tuples are preferable in dealing with entities like origins and
destinations that are related symmetrically."*

### ⚠️Edge case: Flattening an indexed family to a 2-D set with `setof`

When a constraint or operator needs all `(i,j)` pairs across all components of an indexed
family simultaneously, use `setof` to produce a flat 2-D set. The `cross` operator cannot
be applied to an indexed family directly.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/Mihailby/AMPLKnowledge/blob/main/examples/1_1_1__3.ipynb)

In [ ]:
set NODES;                          # All nodes
set ARCS {i in NODES} within NODES; # ARCS[i] = nodes reachable from i

# Flatten indexed family into one 2-D set of pairs
set ALL_ARCS = setof {i in NODES, j in ARCS[i]} (i,j);

# Check whether arc (u,v) exists
param arc_exists {u in NODES, v in NODES} =
    if (u,v) in ALL_ARCS then 1 else 0;

#### amplpy equivalents

In amplpy, sets are accessed via `ampl.set[name]` (equivalent to `ampl.get_set(name)`),
which returns an `amplpy.Set` object.

| Operation | amplpy syntax | Notes |
|---|---|---|
| Assign members (list) | `ampl.set["PROD"] = ["bands", "coils", "plate"]` | Set must be declared first |
| Assign members (method) | `ampl.get_set("PROD").set_values(["bands", "coils", "plate"])` | Explicit method form |
| Read members | `list(ampl.set["PROD"].members())` | Returns a Python list |
| Get cardinality | `ampl.set["PROD"].size()` | Returns an integer |
| Raw AMPL string | `ampl.eval("set PROD := {'bands', 'coils'};" )` | Must include `:=` and proper AMPL syntax |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/Mihailby/AMPLKnowledge/blob/main/examples/1_1_1__4.ipynb)

In [ ]:
ampl.eval("set PROD;")

# Pattern 1: property assignment вЂ” most Pythonic
ampl.set["PROD"] = ["bands", "coils", "plate"]

# Retrieve members and cardinality
members = list(ampl.set["PROD"].members())   # ['bands', 'coils', 'plate']
n       = ampl.set["PROD"].size()             # 3

print("members:", members)
print("size:   ", n)

# Pattern 2: method form
ampl.eval("set NUTR;")
ampl.get_set("NUTR").set_values(["protein", "fat", "sodium", "cal"])
print("NUTR size:", ampl.set["NUTR"].size())  # 4

#### See also

- [§1.1 — Sets](<1_1 Sets.ipynb>) — General set syntax and `SEXPR`.
- [§1.1.2 — Set Declaration vs. Definition](<1_1_2 Set Declaration vs. Definition.ipynb>) — Difference between `set PROD;`, `set PROD = ...`, and data assignment with `:=`.
- [§1.1.3 — Subsets and Supersets](<1_1_3 Subsets and Supersets.ipynb>) — More practice with `within` restrictions.
- [§1.1.4 — Unordered Sets](<1_1_4 Unordered Sets.ipynb>) — Default set type and ordering restrictions.

